# FractalNets y la Autosimilitud

**Curso:** Redes Neuronales Profundas  
**Referencia principal:** Larsson, G., Maire, M. & Shakhnarovich, G. (2017). *FractalNet: Ultra-Deep Neural Networks without Residuals*. ICLR 2017.

---

## Introducción

Las **FractalNets** nacen en el mundo de las redes neuronales convolucionales (CNNs) como una propuesta radical: demostrar que es posible entrenar redes **ultra-profundas** (40+ capas) **sin conexiones residuales**.

Mientras que las ResNets (He et al., 2016) resolvieron el problema de la degradación por profundidad añadiendo atajos de identidad (*skip connections*), las FractalNets proponen una solución puramente **estructural**: una topología de red construida mediante una **regla de expansión recursiva y autosimilar**, inspirada en la geometría fractal.

En este notebook exploraremos:
1. Cómo la regla de expansión fractal genera profundidad de forma exponencial
2. Cómo el *Join Layer* difiere fundamentalmente de la suma residual
3. Cómo implementar y visualizar un bloque fractal desde cero con NumPy
4. La propiedad *anytime*: cada columna es un predictor independiente

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

# Configuración global de matplotlib para figuras limpias
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 11,
    'figure.dpi': 120
})

---

## 1. Ruptura con el paradigma residual

### El problema

Las redes profundas tradicionales apilan capas secuencialmente. Al aumentar la profundidad, el gradiente se desvanece y la red deja de aprender. Las **ResNets** solucionaron esto sumando la entrada original a la salida de cada bloque:

$$y = F(x) + x \quad \text{(conexión residual)}$$

Esto funciona, pero introduce un **sesgo estructural**: la señal de identidad $x$ tiene un estatus privilegiado. El residuo $F(x)$ se aprende "sobre" ella.

### La alternativa fractal

Las FractalNets proponen algo diferente: en lugar de apilar capas y añadir atajos, **generan profundidad mediante una regla recursiva autosimilar**. No hay señal de identidad privilegiada; todos los caminos paralelos son salidas de operaciones aprendidas.

Veamos cómo se construye esta estructura paso a paso.

---

## 2. La regla de expansión estructural — Visualización

La idea central de FractalNet es una **regla de expansión recursiva** que define la topología de la red:

- **Caso base ($C=1$):** Una sola capa de procesamiento (convolucional en el paper original, pero puede ser cualquier tipo de capa, por ejemplo, agregación de vecinos en GNNs).
- **Expansión ($C \to C+1$):** El flujo de información se divide en:
  - **Ruta en serie:** el bloque $f_C$ se aplica dos veces consecutivamente (duplica la profundidad)
  - **Ruta en paralelo:** una sola operación (atajo superficial)
  - Ambas rutas convergen en un **Join Layer** que calcula la media elemento a elemento

La siguiente celda reproduce visualmente la Figura 1 del paper de FractalNet, mostrando la topología para $C=1, 2, 3, 4$.

In [ ]:
# =============================================================================
# VISUALIZACIÓN DE LA REGLA DE EXPANSIÓN FRACTAL (Figura 1 del paper)
# =============================================================================
# Esta es la visualización clave del notebook. Dibuja la topología de red
# para cada nivel de expansión C=1 hasta C=4.
#
# Cada "columna" de la FractalNet tiene una profundidad diferente.
# La columna más a la izquierda es la más profunda (2^{C-1} capas),
# la columna más a la derecha es la más superficial (1 capa).
# =============================================================================

# --- Paleta de colores para las columnas ---
# Cada columna tiene un color distinto para facilitar el seguimiento visual
COLUMN_COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']
JOIN_COLOR = '#607D8B'     # Gris azulado para los Join Layers
BG_COLOR = '#FAFAFA'       # Fondo suave


def build_fractal_topology(C):
    """
    Construye la topología de un bloque fractal de nivel C.
    
    Retorna una estructura recursiva que describe los nodos y conexiones.
    Cada nodo es un diccionario con:
      - 'type': 'conv' o 'join'
      - 'column': a qué columna pertenece (para colorear)
      - 'children': lista de sub-topologías que alimentan este nodo
    
    La topología se construye de SALIDA a ENTRADA (de arriba a abajo en el dibujo).
    """
    if C == 1:
        # Caso base: una sola capa conv
        return {'type': 'conv', 'column': 0, 'depth': 1}
    else:
        # Ruta profunda: f_C compuesto consigo mismo (f_C ∘ f_C)
        deep_first = build_fractal_topology(C - 1)   # primera aplicación
        deep_second = build_fractal_topology(C - 1)   # segunda aplicación
        # Ruta superficial: una sola conv
        shallow = {'type': 'conv', 'column': C - 1, 'depth': 1}
        # Join layer que combina ambas rutas
        return {
            'type': 'join',
            'deep_path': [deep_first, deep_second],  # secuencial
            'shallow_path': shallow,
            'depth': 2 * deep_first.get('depth', 1) if C > 1 else 1
        }


def layout_fractal(C, x_offset=0, y_offset=0, x_scale=1.0):
    """
    Calcula las posiciones (x, y) de cada nodo para dibujar el bloque fractal.
    
    Retorna una lista de nodos con posiciones y una lista de conexiones (aristas).
    El flujo va de ABAJO (entrada) hacia ARRIBA (salida).
    """
    nodes = []  # Lista de {'x', 'y', 'type', 'column', 'label'}
    edges = []  # Lista de ((x1,y1), (x2,y2), color)
    
    # Profundidad máxima del bloque (determina la altura del dibujo)
    max_depth = 2 ** (C - 1)
    
    def _layout(c, y_start, y_end, x_center, column_offset=0):
        """
        Función recursiva que posiciona los nodos.
        
        Parámetros:
          c: nivel de expansión actual
          y_start: posición y inferior (entrada)
          y_end: posición y superior (salida)
          x_center: posición x central
          column_offset: desplazamiento para asignar colores de columna
        
        Retorna: (x, y) del nodo de salida de este sub-bloque
        """
        if c == 1:
            # Caso base: un solo nodo conv en el centro vertical
            y_mid = (y_start + y_end) / 2
            col_idx = column_offset
            nodes.append({
                'x': x_center, 'y': y_mid,
                'type': 'conv',
                'column': col_idx,
                'label': 'conv'
            })
            return (x_center, y_mid)
        
        # --- Ruta profunda (izquierda): f_{c-1} ∘ f_{c-1} ---
        y_mid_deep = (y_start + y_end) / 2
        deep_x = x_center - x_scale * 0.6 * (0.5 ** (C - c))  # desplazar a la izquierda
        
        # Primera mitad: f_{c-1} sobre la mitad inferior
        out1 = _layout(c - 1, y_start, y_mid_deep, deep_x, column_offset)
        # Segunda mitad: f_{c-1} sobre la mitad superior
        out2 = _layout(c - 1, y_mid_deep, y_end, deep_x, column_offset)
        
        # Conexión entre las dos mitades de la ruta profunda
        col_color = COLUMN_COLORS[column_offset % len(COLUMN_COLORS)]
        edges.append((out1, (out2[0], out2[1]), col_color))
        
        # --- Ruta superficial (derecha): una sola conv ---
        shallow_x = x_center + x_scale * 0.6 * (0.5 ** (C - c))
        shallow_col = column_offset + (c - 1)  # columna más superficial
        y_shallow = (y_start + y_end) / 2
        nodes.append({
            'x': shallow_x, 'y': y_shallow,
            'type': 'conv',
            'column': shallow_col,
            'label': 'conv'
        })
        
        # --- Join Layer (arriba) ---
        y_join = y_end
        nodes.append({
            'x': x_center, 'y': y_join,
            'type': 'join',
            'column': -1,
            'label': 'Join'
        })
        
        # Conexiones hacia el join
        shallow_color = COLUMN_COLORS[shallow_col % len(COLUMN_COLORS)]
        edges.append(((shallow_x, y_shallow), (x_center, y_join), shallow_color))
        edges.append((out2, (x_center, y_join), col_color))
        
        return (x_center, y_join)
    
    _layout(C, y_offset, y_offset + max_depth, x_offset)
    return nodes, edges, max_depth


def draw_fractal_block(ax, C, x_offset=0, title=True):
    """
    Dibuja un bloque fractal de nivel C en el axes dado.
    """
    nodes, edges, max_depth = layout_fractal(C, x_offset=0, y_offset=0, x_scale=1.8)
    
    # Dibujar aristas (conexiones)
    for (x1, y1), (x2, y2), color in edges:
        ax.plot([x1, x2], [y1, y2], '-', color=color, linewidth=2.0, alpha=0.7, zorder=1)
    
    # Dibujar nodos
    for node in nodes:
        x, y = node['x'], node['y']
        if node['type'] == 'conv':
            color = COLUMN_COLORS[node['column'] % len(COLUMN_COLORS)]
            box = FancyBboxPatch(
                (x - 0.18, y - 0.18), 0.36, 0.36,
                boxstyle="round,pad=0.05",
                facecolor=color, edgecolor='white',
                linewidth=1.5, alpha=0.9, zorder=2
            )
            ax.add_patch(box)
        elif node['type'] == 'join':
            # Join layers se dibujan como diamantes
            diamond_size = 0.15
            diamond = plt.Polygon([
                (x, y + diamond_size),
                (x + diamond_size, y),
                (x, y - diamond_size),
                (x - diamond_size, y)
            ], facecolor=JOIN_COLOR, edgecolor='white', linewidth=1.5, zorder=2)
            ax.add_patch(diamond)
    
    if title:
        ax.set_title(f'C = {C}\nProf. = {2**(C-1)}', fontsize=13, fontweight='bold', pad=10)
    
    # Configuración del axes
    margin = 0.5
    all_x = [n['x'] for n in nodes]
    all_y = [n['y'] for n in nodes]
    if all_x and all_y:
        ax.set_xlim(min(all_x) - margin, max(all_x) + margin)
        ax.set_ylim(min(all_y) - margin, max(all_y) + margin)
    ax.set_aspect('equal')
    ax.axis('off')


# --- Figura principal: Regla de expansión para C=1,2,3,4 ---
fig, axes = plt.subplots(1, 4, figsize=(18, 7))
fig.suptitle('Regla de Expansión Fractal — Topología para C = 1, 2, 3, 4',
             fontsize=16, fontweight='bold', y=1.02)

for i, C in enumerate([1, 2, 3, 4]):
    draw_fractal_block(axes[i], C)

# Leyenda
legend_elements = [
    mpatches.Patch(facecolor=COLUMN_COLORS[0], label='Columna C (más profunda)'),
    mpatches.Patch(facecolor=COLUMN_COLORS[1], label='Columna 2'),
    mpatches.Patch(facecolor=COLUMN_COLORS[2], label='Columna 3'),
    mpatches.Patch(facecolor=COLUMN_COLORS[3], label='Columna 1 (más superficial)'),
    mpatches.Patch(facecolor=JOIN_COLOR, label='Join Layer (media)')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5,
           fontsize=10, frameon=True, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.show()

### Lectura del diagrama

- **Cuadrados de colores**: capas convolucionales (o cualquier capa de procesamiento). Cada color representa una *columna* de profundidad diferente.
- **Diamantes grises**: *Join Layers* que promedian las señales entrantes.
- **$C=1$**: Una sola capa (caso base).
- **$C=2$**: Dos rutas — una profunda (2 convs en serie, azul) y una superficial (1 conv, verde) — unidas por un Join.
- **$C=3$**: La ruta profunda de $C=2$ se duplica, y se añade una nueva columna superficial (naranja). Profundidad máxima = 4.
- **$C=4$**: La estructura se expande de nuevo. Profundidad máxima = 8.

Observa la **autosimilitud**: cada sub-bloque dentro de la ruta profunda tiene exactamente la misma estructura que el bloque completo del nivel anterior. Esto es lo que hace *fractal* a la FractalNet.

---

## 3. Formalización matemática

Podemos expresar la regla de expansión de forma compacta:

$$
\boxed{
\begin{aligned}
f_1(z) &= \text{conv}(z) \\
f_{C+1}(z) &= \text{Join}\big((f_C \circ f_C)(z),\; \text{conv}(z)\big)
\end{aligned}
}
$$

Donde:
- $(f_C \circ f_C)(z)$ significa aplicar $f_C$ dos veces en secuencia: primero $f_C(z)$, luego $f_C$ sobre ese resultado
- $\text{conv}(z)$ es una única operación de procesamiento
- $\text{Join}(a, b) = \frac{a + b}{2}$ (media elemento a elemento)

### Profundidad por bloque

La profundidad máxima (largo del camino más profundo) en un bloque de nivel $C$ es:

$$\text{prof}(C) = 2^{C-1}$$

### Profundidad total de la red

Al apilar $B$ bloques fractales con capas de *pooling* entre ellos:

$$\text{Prof. total} = B \cdot 2^{C-1}$$

Por ejemplo, con $B=5$ bloques y $C=4$ columnas: $5 \times 2^3 = 40$ capas convolucionales.

---

## 4. Implementación de un bloque fractal con NumPy

A continuación implementamos un bloque fractal funcional que procesa una señal 1D. Cada "conv" será una transformación lineal simple ($Wx + b$) seguida de ReLU, para mantener el ejemplo accesible.

La clase `FractalBlock` construye recursivamente la estructura $f_C$ y permite procesar datos reales a través de ella.

In [ ]:
# =============================================================================
# IMPLEMENTACIÓN DE UN BLOQUE FRACTAL EN NUMPY
# =============================================================================
# Simulamos las "convoluciones" con transformaciones lineales + ReLU.
# Lo importante aquí es la ESTRUCTURA recursiva, no el tipo de operación.
# =============================================================================

def relu(x):
    """Función de activación ReLU."""
    return np.maximum(0, x)


def join_layer(inputs):
    """
    Join Layer: calcula la media elemento a elemento de las entradas.
    
    A diferencia de una conexión residual (que suma identidad + residuo),
    aquí TODAS las entradas son salidas de operaciones aprendidas.
    Ninguna tiene estatus privilegiado.
    """
    return np.mean(inputs, axis=0)


class SimpleConv:
    """
    Capa de procesamiento simple: transformación lineal + ReLU.
    Simula una convolución para propósitos didácticos.
    """
    def __init__(self, dim, name="conv"):
        # Inicialización Xavier para estabilidad numérica
        scale = np.sqrt(2.0 / dim)
        self.W = np.random.randn(dim, dim) * scale
        self.b = np.zeros(dim)
        self.name = name
    
    def __call__(self, x):
        return relu(self.W @ x + self.b)


class FractalBlock:
    """
    Bloque Fractal de nivel C.
    
    Implementa la regla recursiva:
      f_1(z) = conv(z)
      f_{C+1}(z) = Join( (f_C ∘ f_C)(z), conv(z) )
    
    Además, registra las salidas de cada columna para visualización.
    """
    def __init__(self, dim, C, column_offset=0):
        """
        Parámetros:
          dim: dimensión de la señal de entrada
          C: nivel de expansión fractal (número de columnas)
          column_offset: offset para numerar las columnas correctamente
        """
        self.C = C
        self.dim = dim
        self.column_outputs = {}  # {columna: salida} para inspección
        
        if C == 1:
            # Caso base: una sola capa
            self.conv = SimpleConv(dim, name=f"conv_col{column_offset+1}")
            self.column_idx = column_offset + 1
        else:
            # Ruta profunda: dos sub-bloques f_{C-1} en serie
            self.deep_block_1 = FractalBlock(dim, C - 1, column_offset)
            self.deep_block_2 = FractalBlock(dim, C - 1, column_offset)
            # Ruta superficial: una sola conv
            self.shallow_conv = SimpleConv(dim, name=f"conv_col{column_offset + C}")
            self.shallow_column_idx = column_offset + C
    
    def __call__(self, z):
        """
        Procesa la entrada z a través del bloque fractal.
        Retorna la salida y almacena las salidas intermedias por columna.
        """
        self.column_outputs = {}
        
        if self.C == 1:
            # Caso base
            out = self.conv(z)
            self.column_outputs[self.column_idx] = out.copy()
            return out
        else:
            # --- Ruta profunda: (f_C ∘ f_C)(z) ---
            intermediate = self.deep_block_1(z)        # Primera aplicación
            deep_out = self.deep_block_2(intermediate)  # Segunda aplicación
            
            # Recopilar salidas de columnas del camino profundo
            self.column_outputs.update(self.deep_block_1.column_outputs)
            self.column_outputs.update(self.deep_block_2.column_outputs)
            
            # --- Ruta superficial: conv(z) ---
            shallow_out = self.shallow_conv(z)
            self.column_outputs[self.shallow_column_idx] = shallow_out.copy()
            
            # --- Join Layer: media de ambas rutas ---
            joined = join_layer(np.array([deep_out, shallow_out]))
            return joined


# --- Demostración: procesar una señal 1D ---
np.random.seed(42)

dim = 8          # Dimensión de la señal
C = 3            # Nivel de expansión fractal (3 columnas)

# Crear señal de entrada
z = np.random.randn(dim)
print("=" * 60)
print(f"BLOQUE FRACTAL — C={C} (profundidad máx. = {2**(C-1)})")
print("=" * 60)
print(f"\nEntrada z (dim={dim}):")
print(f"  {np.round(z, 3)}")

# Crear y ejecutar el bloque fractal
fractal = FractalBlock(dim, C)
output = fractal(z)

print(f"\nSalida del bloque fractal:")
print(f"  {np.round(output, 3)}")

print(f"\n--- Salidas por columna ---")
for col_idx in sorted(fractal.column_outputs.keys()):
    col_out = fractal.column_outputs[col_idx]
    depth = 2 ** (col_idx - 1)  # Profundidad de la columna
    print(f"  Columna {col_idx} (profundidad {depth}): {np.round(col_out, 3)}")
    print(f"    Norma L2: {np.linalg.norm(col_out):.4f}")

### Observaciones

Nota cómo las columnas de diferentes profundidades producen salidas distintas:
- La **Columna $C$** (más profunda) pasa por $2^{C-1}$ transformaciones sucesivas
- La **Columna 1** (más superficial) pasa por solo 1 transformación

**Convención de numeración:** A lo largo de estos notebooks, la Columna 1 siempre es la más superficial (1 capa) y la Columna $C$ la más profunda ($2^{C-1}$ capas). Esta convención es consistente con el código, donde la columna $k$ tiene $2^{k-1}$ capas.

---

## 5. El Join Layer vs. la conexión residual

Veamos con un ejemplo numérico concreto la diferencia entre el **Join Layer** de FractalNet y la **suma residual** de ResNet.

- En **ResNet**: $y = F(x) + x$ — la identidad $x$ tiene estatus privilegiado
- En **FractalNet**: $y = \text{Join}(\text{ruta}_1, \text{ruta}_2) = \frac{\text{ruta}_1 + \text{ruta}_2}{2}$ — ninguna ruta es privilegiada

In [ ]:
# =============================================================================
# COMPARACIÓN: JOIN LAYER (FractalNet) vs. SUMA RESIDUAL (ResNet)
# =============================================================================

np.random.seed(123)

# Señal de entrada
x = np.array([1.0, 2.0, 3.0, 4.0])

# Simular salidas de capas de procesamiento
# (En la práctica estas son salidas de convoluciones aprendidas)
ruta_profunda = np.array([0.5, 1.2, 0.8, 2.1])    # Salida de ruta profunda
ruta_superficial = np.array([0.9, 1.8, 2.5, 3.2])  # Salida de ruta superficial

# --- FractalNet: Join Layer (media) ---
join_output = (ruta_profunda + ruta_superficial) / 2

# --- ResNet: Suma residual ---
# En ResNet, F(x) es el residuo y x es la identidad
residuo = ruta_profunda  # Solo hay una ruta aprendida
resnet_output = residuo + x  # y = F(x) + x

print("=" * 60)
print("COMPARACIÓN: FractalNet Join vs. ResNet Skip Connection")
print("=" * 60)

print(f"\nEntrada x:              {x}")
print(f"Ruta profunda (conv):   {ruta_profunda}")
print(f"Ruta superficial (conv): {ruta_superficial}")

print(f"\n--- FractalNet (Join = media) ---")
print(f"Join(profunda, superficial) = (profunda + superficial) / 2")
print(f"= ({ruta_profunda} + {ruta_superficial}) / 2")
print(f"= {join_output}")
print(f"Norma: {np.linalg.norm(join_output):.4f}")

print(f"\n--- ResNet (Suma residual) ---")
print(f"y = F(x) + x  (identidad tiene estatus privilegiado)")
print(f"= {residuo} + {x}")
print(f"= {resnet_output}")
print(f"Norma: {np.linalg.norm(resnet_output):.4f}")

print(f"\n--- Diferencia clave ---")
print(f"En ResNet: si F(x) ≈ 0, la salida ≈ x (la identidad domina)")
print(f"En FractalNet: ambas rutas son conv, ninguna es identidad.")
print(f"El Join estabiliza la magnitud al promediar.")

# --- Visualización ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: FractalNet Join
ax = axes[0]
bar_width = 0.25
indices = np.arange(len(x))
ax.bar(indices - bar_width, ruta_profunda, bar_width, color=COLUMN_COLORS[0],
       label='Ruta profunda (conv)', alpha=0.8)
ax.bar(indices, ruta_superficial, bar_width, color=COLUMN_COLORS[1],
       label='Ruta superficial (conv)', alpha=0.8)
ax.bar(indices + bar_width, join_output, bar_width, color=JOIN_COLOR,
       label='Join (media)', alpha=0.8)
ax.set_xlabel('Dimensión')
ax.set_ylabel('Valor')
ax.set_title('FractalNet: Join Layer\n(media de rutas conv)', fontweight='bold')
ax.set_xticks(indices)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Panel derecho: ResNet Skip
ax = axes[1]
ax.bar(indices - bar_width, x, bar_width, color='#78909C',
       label='Identidad (x)', alpha=0.8)
ax.bar(indices, residuo, bar_width, color=COLUMN_COLORS[0],
       label='Residuo F(x)', alpha=0.8)
ax.bar(indices + bar_width, resnet_output, bar_width, color='#E91E63',
       label='F(x) + x', alpha=0.8)
ax.set_xlabel('Dimensión')
ax.set_ylabel('Valor')
ax.set_title('ResNet: Conexión Residual\n(identidad + residuo)', fontweight='bold')
ax.set_xticks(indices)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Punto clave

La diferencia no es simplemente aritmética (media vs. suma). Es **estructural**:

| Aspecto | ResNet | FractalNet |
|---------|--------|------------|
| Señal privilegiada | Sí (identidad $x$) | No |
| Entradas al punto de unión | Identidad + residuo aprendido | Múltiples rutas aprendidas |
| Si la parte aprendida falla | La identidad preserva la señal | El promedio estabiliza |
| Filosofía | "Aprende la diferencia" | "Múltiples caminos independientes" |

**Nota técnica:** En la comparación numérica anterior, tratamos las salidas de las rutas como vectores estáticos para ilustrar el efecto aritmético de cada operación (control de magnitud del promedio vs. la suma). En una ResNet real entrenada, los pesos se optimizarían *asumiendo* que la salida se sumará a la identidad — el bloque aprendería literalmente el **residuo** (lo que falta), no una transformación completa. Esta salvedad no invalida la comparación, pero es importante para entender que la diferencia fundamental es de *diseño arquitectónico*, no solo de operación aritmética.

---

## 6. Crecimiento exponencial de la profundidad

Una propiedad notable de la regla de expansión fractal es que la profundidad crece **exponencialmente** con el número de columnas $C$, mientras que la red completa con $B$ bloques alcanza una profundidad total de $B \cdot 2^{C-1}$.

La siguiente celda visualiza este crecimiento.

In [ ]:
# =============================================================================
# CRECIMIENTO DE LA PROFUNDIDAD: exponencial con C, lineal con B
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel izquierdo: Profundidad por bloque vs C ---
ax = axes[0]
C_values = np.arange(1, 9)  # C = 1 a 8
depth_per_block = 2 ** (C_values - 1)

ax.bar(C_values, depth_per_block, color=COLUMN_COLORS[0], alpha=0.8, edgecolor='white')
for i, (c, d) in enumerate(zip(C_values, depth_per_block)):
    ax.text(c, d + 1, str(d), ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_xlabel('Número de columnas (C)', fontsize=12)
ax.set_ylabel('Profundidad máxima por bloque', fontsize=12)
ax.set_title('Profundidad por bloque: $2^{C-1}$', fontsize=13, fontweight='bold')
ax.set_xticks(C_values)
ax.grid(axis='y', alpha=0.3)

# --- Panel derecho: Profundidad total vs (B, C) ---
ax = axes[1]
B_values = [1, 2, 3, 4, 5]
C_range = np.arange(1, 7)  # C = 1 a 6

for i, B in enumerate(B_values):
    total_depth = B * 2 ** (C_range - 1)
    ax.plot(C_range, total_depth, 'o-', color=COLUMN_COLORS[i % len(COLUMN_COLORS)],
            linewidth=2, markersize=8, label=f'B = {B} bloques')

# Marcar la configuración del paper original (B=5, C=4 → 40 capas)
ax.plot(4, 5 * 2**3, '*', color='red', markersize=20, zorder=5)
ax.annotate('Paper original\nB=5, C=4 → 40 capas',
            xy=(4, 40), xytext=(4.8, 55),
            fontsize=10, fontweight='bold', color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax.set_xlabel('Número de columnas (C)', fontsize=12)
ax.set_ylabel('Profundidad total de la red', fontsize=12)
ax.set_title('Profundidad total: $B \cdot 2^{C-1}$', fontsize=13, fontweight='bold')
ax.set_xticks(C_range)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla resumen
print("\n" + "=" * 50)
print("Tabla: Profundidad total = B × 2^(C-1)")
print("=" * 50)
header = f"{'B \\ C':>8}" + "".join(f"{c:>8}" for c in range(1, 7))
print(header)
print("-" * len(header))
for B in B_values:
    row = f"{B:>8}" + "".join(f"{B * 2**(c-1):>8}" for c in range(1, 7))
    print(row)

### Observaciones

- Con solo $C=4$ columnas y $B=5$ bloques se alcanzan **40 capas** de profundidad
- El crecimiento exponencial permite redes muy profundas con pocos niveles de recursión
- Comparado con una ResNet que simplemente apila 40 bloques residuales, la FractalNet obtiene la misma profundidad pero con **múltiples caminos de diferente longitud** integrados en la estructura

---

## 7. La propiedad *anytime*: columnas como predictores independientes

Una consecuencia fascinante de la topología fractal es la propiedad ***anytime***: cada columna individual de la red puede funcionar como un predictor independiente con diferente profundidad.

- **Columna 1** (la más superficial): utiliza solo 1 capa — inferencia rápida
- **Columna $C$** (la más profunda): utiliza $2^{C-1}$ capas — mayor precisión

Gracias a la regularización *drop-path* durante el entrenamiento (que fuerza a cada columna a ser competente por sí sola), es posible extraer subredes de diferentes profundidades en tiempo de inferencia.

In [ ]:
# =============================================================================
# PROPIEDAD ANYTIME: cada columna produce una salida independiente
# =============================================================================
# Demostración: procesamos una señal sinusoidal a través de un bloque fractal
# C=4 y observamos cómo cada columna (de diferente profundidad) la transforma.
# =============================================================================

np.random.seed(7)

dim = 32   # Dimensión más alta para ver patrones más claros
C = 4      # 4 columnas → profundidades 8, 4, 2, 1

# Señal de entrada: combinación de sinusoides (simula un patrón estructurado)
t = np.linspace(0, 2 * np.pi, dim)
z = np.sin(t) + 0.5 * np.sin(3 * t) + 0.1 * np.random.randn(dim)

# Crear y ejecutar el bloque fractal
fractal_block = FractalBlock(dim, C)
final_output = fractal_block(z)

# --- Visualización ---
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Señal de entrada
ax = axes[0, 0]
ax.plot(t, z, 'k-', linewidth=2, label='Entrada z')
ax.fill_between(t, z, alpha=0.1, color='black')
ax.set_title('Señal de entrada', fontsize=12, fontweight='bold')
ax.set_xlabel('t')
ax.legend()
ax.grid(alpha=0.3)

# Salida de cada columna
column_outputs = fractal_block.column_outputs
sorted_cols = sorted(column_outputs.keys())

for i, col_idx in enumerate(sorted_cols):
    row, col = divmod(i + 1, 3)
    ax = axes[row, col]
    
    col_out = column_outputs[col_idx]
    depth = 2 ** (col_idx - 1)
    color = COLUMN_COLORS[(col_idx - 1) % len(COLUMN_COLORS)]
    
    ax.plot(t, col_out, '-', linewidth=2, color=color,
            label=f'Columna {col_idx}')
    ax.fill_between(t, col_out, alpha=0.15, color=color)
    ax.plot(t, z, 'k--', linewidth=0.8, alpha=0.4, label='Entrada')
    ax.set_title(f'Columna {col_idx} — Profundidad = {depth}',
                 fontsize=12, fontweight='bold', color=color)
    ax.set_xlabel('t')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

# Salida final (Join de todas las rutas)
ax = axes[1, 2]
ax.plot(t, final_output, '-', linewidth=2.5, color=JOIN_COLOR,
        label='Salida final (Join)')
ax.fill_between(t, final_output, alpha=0.15, color=JOIN_COLOR)
ax.plot(t, z, 'k--', linewidth=0.8, alpha=0.4, label='Entrada')
ax.set_title('Salida final — Join de todas las rutas',
             fontsize=12, fontweight='bold', color=JOIN_COLOR)
ax.set_xlabel('t')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

fig.suptitle(f'Propiedad Anytime — Bloque Fractal C={C}\n'
             f'Cada columna es un predictor independiente de diferente profundidad',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Estadísticas por columna
print("\n" + "=" * 60)
print("Estadísticas por columna")
print("=" * 60)
print(f"{'Columna':<10} {'Profundidad':<15} {'Norma L2':<15} {'Media':<15} {'Std':<15}")
print("-" * 70)
for col_idx in sorted_cols:
    col_out = column_outputs[col_idx]
    depth = 2 ** (col_idx - 1)
    print(f"{col_idx:<10} {depth:<15} {np.linalg.norm(col_out):<15.4f} "
          f"{np.mean(col_out):<15.4f} {np.std(col_out):<15.4f}")

### Interpretación

Cada columna transforma la señal de entrada de forma diferente según su profundidad:

- Las **columnas profundas** (izquierda) aplican más transformaciones no lineales, extrayendo representaciones más abstractas
- Las **columnas superficiales** (derecha) aplican pocas transformaciones, preservando más de la estructura original
- La **salida final** (Join) promedia todas las perspectivas

En una FractalNet entrenada con *drop-path*, cada columna se entrena para ser un predictor competente. Esto permite **extraer subredes más pequeñas y rápidas** en inferencia sin reentrenar, sacrificando algo de precisión a cambio de velocidad.

---

## 8. Resumen: conexión con el problema de la profundidad

### Lo que hemos visto

1. **Las FractalNets generan profundidad mediante autosimilitud**, no apilando bloques residuales. La regla recursiva $f_{C+1}(z) = \text{Join}((f_C \circ f_C)(z), \text{conv}(z))$ produce profundidad exponencial con pocos niveles de recursión.

2. **El Join Layer no tiene señal privilegiada.** A diferencia de las ResNets (donde la identidad es un "ancla"), en FractalNet todas las entradas al Join son salidas de operaciones aprendidas. Esto estabiliza las activaciones sin depender de atajos de identidad.

3. **La profundidad crece como $2^{C-1}$ por bloque.** Con $B=5$ bloques y $C=4$ columnas se obtienen 40 capas — comparable a las ResNets más profundas de su época.

4. **Cada columna es un predictor independiente** (propiedad *anytime*). Gracias a la regularización *drop-path*, es posible extraer subredes de diferente profundidad sin reentrenar.

### La idea fundamental

La contribución de FractalNet no es una arquitectura específica, sino un **principio de diseño**: la **autosimilitud estructural** como alternativa a las conexiones residuales para habilitar profundidad. Este principio es transferible a otros dominios — como las Redes Neuronales de Grafos (GNNs), donde el problema de la profundidad se manifiesta como *over-smoothing*.

### Referencia

Larsson, G., Maire, M. & Shakhnarovich, G. (2017). *FractalNet: Ultra-Deep Neural Networks without Residuals*. ICLR 2017.